In [ ]:
<head><meta charset="UTF-8"></head><pre style="caret-color: rgb(0, 0, 0); color: rgb(0, 0, 0); font-style: normal; font-variant-caps: normal; font-weight: 400; letter-spacing: normal; orphans: auto; text-align: start; text-indent: 0px; text-transform: none; widows: auto; word-spacing: 0px; -webkit-tap-highlight-color: rgba(26, 26, 26, 0.3); -webkit-text-size-adjust: auto; -webkit-text-stroke-width: 0px; text-decoration: none; overflow-wrap: break-word; white-space: pre-wrap;">&lt;p style="text-align:center"&gt;
    &lt;a href="https://skills.network" target="_blank"&gt;
    &lt;img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"&gt;
    &lt;/a&gt;
&lt;/p&gt;


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab



```python
!pip3 install beautifulsoup4
!pip3 install requests
```

    Requirement already satisfied: beautifulsoup4 in /home/jupyterlab/conda/envs/python/lib/python3.7/site-packages (4.11.1)
    Requirement already satisfied: soupsieve&gt;1.2 in /home/jupyterlab/conda/envs/python/lib/python3.7/site-packages (from beautifulsoup4) (2.3.2.post1)
    Requirement already satisfied: requests in /home/jupyterlab/conda/envs/python/lib/python3.7/site-packages (2.29.0)
    Requirement already satisfied: charset-normalizer&lt;4,&gt;=2 in /home/jupyterlab/conda/envs/python/lib/python3.7/site-packages (from requests) (3.1.0)
    Requirement already satisfied: idna&lt;4,&gt;=2.5 in /home/jupyterlab/conda/envs/python/lib/python3.7/site-packages (from requests) (3.4)
    Requirement already satisfied: urllib3&lt;1.27,&gt;=1.21.1 in /home/jupyterlab/conda/envs/python/lib/python3.7/site-packages (from requests) (1.26.15)
    Requirement already satisfied: certifi&gt;=2017.4.17 in /home/jupyterlab/conda/envs/python/lib/python3.7/site-packages (from requests) (2023.5.7)



```python
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd
```

and we will provide some helper functions for you to process web scraped HTML table



```python
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    

```

To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`



```python
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&amp;oldid=1027686922"
```

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.



```python
# use requests.get() method with the provided static_url
# assign the response to a object
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&amp;oldid=1027686922"
response = requests.get(url)

if response.status_code == 200:
    print("Successfully retrieved the page")
else:
    print("Failed to retrieve the page")
```

    Successfully retrieved the page



```python
import requests
from bs4 import BeautifulSoup

# Step 1: Fetch the HTML content
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&amp;oldid=1027686922"
response = requests.get(url)

# Step 2: Check the response status
if response.status_code == 200:
    print("Successfully retrieved the page")
    
    # Step 3: Create a BeautifulSoup object
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # You can now use 'soup' to navigate and search the HTML tree
   # print(soup.prettify())  # Optional: Print the formatted HTML
else:
    print("Failed to retrieve the page")
```

    Successfully retrieved the page


Create a `BeautifulSoup` object from the HTML `response`


Print the page title to verify if the `BeautifulSoup` object was created properly 



```python
# Use soup.title attribute
print(soup.title.string)
```

    List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


Starting from the third table is our target table contains the actual launch records.



```python
import requests
from bs4 import BeautifulSoup

# 指定维基百科页面的 URL
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&amp;oldid=1027686922"

# 发送 GET 请求
response = requests.get(url)

# 检查请求是否成功
if response.status_code == 200:
    # 创建 BeautifulSoup 对象
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 找到所有的表格
    tables = soup.find_all('table', {'class': 'wikitable'})
    
    # 打印找到的表格数量
    print(f"找到的表格数量: {len(tables)}")
    
    # 选择第三个表格（索引为2）
    target_table = tables[2]  # 从第三个表格开始
    
    # 提取列名
    headers = [header.text.strip() for header in target_table.find_all('th')]
    
    # 打印列名
    print("列名:")
    for header in headers:
        print(header)
else:
    print(f"请求失败，状态码: {response.status_code}")
```

    找到的表格数量: 13
    列名:
    Flight No.
    Date andtime (UTC)
    Version,Booster[b]
    Launch site
    Payload[c]
    Payload mass
    Orbit
    Customer
    Launchoutcome
    Boosterlanding
    14
    15
    16
    17
    18
    19
    20



```python
# Let's print the third table and check its content
first_launch_table = tables[2]
print(first_launch_table)
```

    &lt;table class="wikitable plainrowheaders collapsible" style="width: 100%;"&gt;
    &lt;tbody&gt;&lt;tr&gt;
    &lt;th scope="col"&gt;Flight No.
    &lt;/th&gt;
    &lt;th scope="col"&gt;Date and&lt;br/&gt;time (&lt;a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time"&gt;UTC&lt;/a&gt;)
    &lt;/th&gt;
    &lt;th scope="col"&gt;&lt;a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters"&gt;Version,&lt;br/&gt;Booster&lt;/a&gt; &lt;sup class="reference" id="cite_ref-booster_11-0"&gt;&lt;a href="#cite_note-booster-11"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;b&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/th&gt;
    &lt;th scope="col"&gt;Launch site
    &lt;/th&gt;
    &lt;th scope="col"&gt;Payload&lt;sup class="reference" id="cite_ref-Dragon_12-0"&gt;&lt;a href="#cite_note-Dragon-12"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;c&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/th&gt;
    &lt;th scope="col"&gt;Payload mass
    &lt;/th&gt;
    &lt;th scope="col"&gt;Orbit
    &lt;/th&gt;
    &lt;th scope="col"&gt;Customer
    &lt;/th&gt;
    &lt;th scope="col"&gt;Launch&lt;br/&gt;outcome
    &lt;/th&gt;
    &lt;th scope="col"&gt;&lt;a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests"&gt;Booster&lt;br/&gt;landing&lt;/a&gt;
    &lt;/th&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;th rowspan="2" scope="row" style="text-align:center;"&gt;1
    &lt;/th&gt;
    &lt;td&gt;4 June 2010,&lt;br/&gt;18:45
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Falcon_9_v1.0" title="Falcon 9 v1.0"&gt;F9 v1.0&lt;/a&gt;&lt;sup class="reference" id="cite_ref-MuskMay2012_13-0"&gt;&lt;a href="#cite_note-MuskMay2012-13"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;7&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;B0003.1&lt;sup class="reference" id="cite_ref-block_numbers_14-0"&gt;&lt;a href="#cite_note-block_numbers-14"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;8&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Force_Station" title="Cape Canaveral Space Force Station"&gt;CCAFS&lt;/a&gt;,&lt;br/&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Launch_Complex_40" title="Cape Canaveral Space Launch Complex 40"&gt;SLC-40&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Dragon_Spacecraft_Qualification_Unit" title="Dragon Spacecraft Qualification Unit"&gt;Dragon Spacecraft Qualification Unit&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Low_Earth_orbit" title="Low Earth orbit"&gt;LEO&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/SpaceX" title="SpaceX"&gt;SpaceX&lt;/a&gt;
    &lt;/td&gt;
    &lt;td class="table-success" style="background: #9EFF9E; color:black; vertical-align: middle; text-align: center;"&gt;Success
    &lt;/td&gt;
    &lt;td class="table-failure" style="background: #FFC7C7; color:black; vertical-align: middle; text-align: center;"&gt;Failure&lt;sup class="reference" id="cite_ref-ns20110930_15-0"&gt;&lt;a href="#cite_note-ns20110930-15"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;9&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;sup class="reference" id="cite_ref-16"&gt;&lt;a href="#cite_note-16"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;10&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;&lt;small&gt;(parachute)&lt;/small&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;td colspan="9"&gt;First flight of Falcon 9 v1.0.&lt;sup class="reference" id="cite_ref-sfn20100604_17-0"&gt;&lt;a href="#cite_note-sfn20100604-17"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;11&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; Used a boilerplate version of Dragon capsule which was not designed to separate from the second stage.&lt;small&gt;(&lt;a href="#First_flight_of_Falcon_9"&gt;more details below&lt;/a&gt;)&lt;/small&gt; Attempted to recover the first stage by parachuting it into the ocean, but it burned up on reentry, before the parachutes even deployed.&lt;sup class="reference" id="cite_ref-parachute_18-0"&gt;&lt;a href="#cite_note-parachute-18"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;12&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;th rowspan="2" scope="row" style="text-align:center;"&gt;2
    &lt;/th&gt;
    &lt;td&gt;8 December 2010,&lt;br/&gt;15:43&lt;sup class="reference" id="cite_ref-spaceflightnow_Clark_Launch_Report_19-0"&gt;&lt;a href="#cite_note-spaceflightnow_Clark_Launch_Report-19"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;13&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Falcon_9_v1.0" title="Falcon 9 v1.0"&gt;F9 v1.0&lt;/a&gt;&lt;sup class="reference" id="cite_ref-MuskMay2012_13-1"&gt;&lt;a href="#cite_note-MuskMay2012-13"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;7&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;B0004.1&lt;sup class="reference" id="cite_ref-block_numbers_14-1"&gt;&lt;a href="#cite_note-block_numbers-14"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;8&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Force_Station" title="Cape Canaveral Space Force Station"&gt;CCAFS&lt;/a&gt;,&lt;br/&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Launch_Complex_40" title="Cape Canaveral Space Launch Complex 40"&gt;SLC-40&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/SpaceX_Dragon" title="SpaceX Dragon"&gt;Dragon&lt;/a&gt; &lt;a class="mw-redirect" href="/wiki/COTS_Demo_Flight_1" title="COTS Demo Flight 1"&gt;demo flight C1&lt;/a&gt;&lt;br/&gt;(Dragon C101)
    &lt;/td&gt;
    &lt;td&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Low_Earth_orbit" title="Low Earth orbit"&gt;LEO&lt;/a&gt; (&lt;a href="/wiki/International_Space_Station" title="International Space Station"&gt;ISS&lt;/a&gt;)
    &lt;/td&gt;
    &lt;td&gt;&lt;style data-mw-deduplicate="TemplateStyles:r1126788409"&gt;.mw-parser-output .plainlist ol,.mw-parser-output .plainlist ul{line-height:inherit;list-style:none;margin:0;padding:0}.mw-parser-output .plainlist ol li,.mw-parser-output .plainlist ul li{margin-bottom:0}&lt;/style&gt;&lt;div class="plainlist"&gt;
    &lt;ul&gt;&lt;li&gt;&lt;a href="/wiki/NASA" title="NASA"&gt;NASA&lt;/a&gt; (&lt;a href="/wiki/Commercial_Orbital_Transportation_Services" title="Commercial Orbital Transportation Services"&gt;COTS&lt;/a&gt;)&lt;/li&gt;
    &lt;li&gt;&lt;a href="/wiki/National_Reconnaissance_Office" title="National Reconnaissance Office"&gt;NRO&lt;/a&gt;&lt;/li&gt;&lt;/ul&gt;
    &lt;/div&gt;
    &lt;/td&gt;
    &lt;td class="table-success" style="background: #9EFF9E; color:black; vertical-align: middle; text-align: center;"&gt;Success&lt;sup class="reference" id="cite_ref-ns20110930_15-1"&gt;&lt;a href="#cite_note-ns20110930-15"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;9&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td class="table-failure" style="background: #FFC7C7; color:black; vertical-align: middle; text-align: center;"&gt;Failure&lt;sup class="reference" id="cite_ref-ns20110930_15-2"&gt;&lt;a href="#cite_note-ns20110930-15"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;9&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;sup class="reference" id="cite_ref-20"&gt;&lt;a href="#cite_note-20"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;14&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;&lt;small&gt;(parachute)&lt;/small&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;td colspan="9"&gt;Maiden flight of &lt;a class="mw-redirect" href="/wiki/Dragon_capsule" title="Dragon capsule"&gt;Dragon capsule&lt;/a&gt;, consisting of over 3 hours of testing thruster maneuvering and reentry.&lt;sup class="reference" id="cite_ref-spaceflightnow_Clark_unleashing_Dragon_21-0"&gt;&lt;a href="#cite_note-spaceflightnow_Clark_unleashing_Dragon-21"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;15&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; Attempted to recover the first stage by parachuting it into the ocean, but it disintegrated upon reentry, before the parachutes were deployed.&lt;sup class="reference" id="cite_ref-parachute_18-1"&gt;&lt;a href="#cite_note-parachute-18"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;12&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; &lt;small&gt;(&lt;a href="#COTS_demo_missions"&gt;more details below&lt;/a&gt;)&lt;/small&gt; It also included two &lt;a href="/wiki/CubeSat" title="CubeSat"&gt;CubeSats&lt;/a&gt;,&lt;sup class="reference" id="cite_ref-NRO_Taps_Boeing_for_Next_Batch_of_CubeSats_22-0"&gt;&lt;a href="#cite_note-NRO_Taps_Boeing_for_Next_Batch_of_CubeSats-22"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;16&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; and a wheel of &lt;a href="/wiki/Brou%C3%A8re" title="Brouère"&gt;Brouère&lt;/a&gt; cheese.
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;th rowspan="2" scope="row" style="text-align:center;"&gt;3
    &lt;/th&gt;
    &lt;td&gt;22 May 2012,&lt;br/&gt;07:44&lt;sup class="reference" id="cite_ref-BBC_new_era_23-0"&gt;&lt;a href="#cite_note-BBC_new_era-23"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;17&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Falcon_9_v1.0" title="Falcon 9 v1.0"&gt;F9 v1.0&lt;/a&gt;&lt;sup class="reference" id="cite_ref-MuskMay2012_13-2"&gt;&lt;a href="#cite_note-MuskMay2012-13"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;7&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;B0005.1&lt;sup class="reference" id="cite_ref-block_numbers_14-2"&gt;&lt;a href="#cite_note-block_numbers-14"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;8&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Force_Station" title="Cape Canaveral Space Force Station"&gt;CCAFS&lt;/a&gt;,&lt;br/&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Launch_Complex_40" title="Cape Canaveral Space Launch Complex 40"&gt;SLC-40&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/SpaceX_Dragon" title="SpaceX Dragon"&gt;Dragon&lt;/a&gt; &lt;a class="mw-redirect" href="/wiki/Dragon_C2%2B" title="Dragon C2+"&gt;demo flight C2+&lt;/a&gt;&lt;sup class="reference" id="cite_ref-C2_24-0"&gt;&lt;a href="#cite_note-C2-24"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;18&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;(Dragon C102)
    &lt;/td&gt;
    &lt;td&gt;525 kg (1,157 lb)&lt;sup class="reference" id="cite_ref-25"&gt;&lt;a href="#cite_note-25"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;19&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Low_Earth_orbit" title="Low Earth orbit"&gt;LEO&lt;/a&gt; (&lt;a href="/wiki/International_Space_Station" title="International Space Station"&gt;ISS&lt;/a&gt;)
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/NASA" title="NASA"&gt;NASA&lt;/a&gt; (&lt;a href="/wiki/Commercial_Orbital_Transportation_Services" title="Commercial Orbital Transportation Services"&gt;COTS&lt;/a&gt;)
    &lt;/td&gt;
    &lt;td class="table-success" style="background: #9EFF9E; color:black; vertical-align: middle; text-align: center;"&gt;Success&lt;sup class="reference" id="cite_ref-26"&gt;&lt;a href="#cite_note-26"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;20&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td class="table-noAttempt" style="background: #EEE; color:black; vertical-align: middle; white-space: nowrap; text-align: center;"&gt;No attempt
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;td colspan="9"&gt;Dragon spacecraft demonstrated a series of tests before it was allowed to approach the &lt;a href="/wiki/International_Space_Station" title="International Space Station"&gt;International Space Station&lt;/a&gt;. Two days later, it became the first commercial spacecraft to board the ISS.&lt;sup class="reference" id="cite_ref-BBC_new_era_23-1"&gt;&lt;a href="#cite_note-BBC_new_era-23"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;17&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; &lt;small&gt;(&lt;a href="#COTS_demo_missions"&gt;more details below&lt;/a&gt;)&lt;/small&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;th rowspan="3" scope="row" style="text-align:center;"&gt;4
    &lt;/th&gt;
    &lt;td rowspan="2"&gt;8 October 2012,&lt;br/&gt;00:35&lt;sup class="reference" id="cite_ref-SFN_LLog_27-0"&gt;&lt;a href="#cite_note-SFN_LLog-27"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;21&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td rowspan="2"&gt;&lt;a href="/wiki/Falcon_9_v1.0" title="Falcon 9 v1.0"&gt;F9 v1.0&lt;/a&gt;&lt;sup class="reference" id="cite_ref-MuskMay2012_13-3"&gt;&lt;a href="#cite_note-MuskMay2012-13"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;7&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;B0006.1&lt;sup class="reference" id="cite_ref-block_numbers_14-3"&gt;&lt;a href="#cite_note-block_numbers-14"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;8&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td rowspan="2"&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Force_Station" title="Cape Canaveral Space Force Station"&gt;CCAFS&lt;/a&gt;,&lt;br/&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Launch_Complex_40" title="Cape Canaveral Space Launch Complex 40"&gt;SLC-40&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/SpaceX_CRS-1" title="SpaceX CRS-1"&gt;SpaceX CRS-1&lt;/a&gt;&lt;sup class="reference" id="cite_ref-sxManifest20120925_28-0"&gt;&lt;a href="#cite_note-sxManifest20120925-28"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;22&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;(Dragon C103)
    &lt;/td&gt;
    &lt;td&gt;4,700 kg (10,400 lb)
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Low_Earth_orbit" title="Low Earth orbit"&gt;LEO&lt;/a&gt; (&lt;a href="/wiki/International_Space_Station" title="International Space Station"&gt;ISS&lt;/a&gt;)
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/NASA" title="NASA"&gt;NASA&lt;/a&gt; (&lt;a href="/wiki/Commercial_Resupply_Services" title="Commercial Resupply Services"&gt;CRS&lt;/a&gt;)
    &lt;/td&gt;
    &lt;td class="table-success" style="background: #9EFF9E; color:black; vertical-align: middle; text-align: center;"&gt;Success
    &lt;/td&gt;
    &lt;td rowspan="2" style="background:#ececec; text-align:center;"&gt;&lt;span class="nowrap"&gt;No attempt&lt;/span&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;td&gt;&lt;a href="/wiki/Orbcomm_(satellite)" title="Orbcomm (satellite)"&gt;Orbcomm-OG2&lt;/a&gt;&lt;sup class="reference" id="cite_ref-Orbcomm_29-0"&gt;&lt;a href="#cite_note-Orbcomm-29"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;23&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;172 kg (379 lb)&lt;sup class="reference" id="cite_ref-gunter-og2_30-0"&gt;&lt;a href="#cite_note-gunter-og2-30"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;24&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Low_Earth_orbit" title="Low Earth orbit"&gt;LEO&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Orbcomm" title="Orbcomm"&gt;Orbcomm&lt;/a&gt;
    &lt;/td&gt;
    &lt;td class="table-partial" style="background: #FFB; color:black; vertical-align: middle; text-align: center;"&gt;Partial failure&lt;sup class="reference" id="cite_ref-nyt-20121030_31-0"&gt;&lt;a href="#cite_note-nyt-20121030-31"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;25&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;td colspan="9"&gt;CRS-1 was successful, but the &lt;a href="/wiki/Secondary_payload" title="Secondary payload"&gt;secondary payload&lt;/a&gt; was inserted into an abnormally low orbit and subsequently lost. This was due to one of the nine &lt;a href="/wiki/SpaceX_Merlin" title="SpaceX Merlin"&gt;Merlin engines&lt;/a&gt; shutting down during the launch, and NASA declining a second reignition, as per &lt;a href="/wiki/International_Space_Station" title="International Space Station"&gt;ISS&lt;/a&gt; visiting vehicle safety rules, the primary payload owner is contractually allowed to decline a second reignition. NASA stated that this was because SpaceX could not guarantee a high enough likelihood of the second stage completing the second burn successfully which was required to avoid any risk of secondary payload's collision with the ISS.&lt;sup class="reference" id="cite_ref-OrbcommTotalLoss_32-0"&gt;&lt;a href="#cite_note-OrbcommTotalLoss-32"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;26&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;sup class="reference" id="cite_ref-sn20121011_33-0"&gt;&lt;a href="#cite_note-sn20121011-33"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;27&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;sup class="reference" id="cite_ref-34"&gt;&lt;a href="#cite_note-34"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;28&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;th rowspan="2" scope="row" style="text-align:center;"&gt;5
    &lt;/th&gt;
    &lt;td&gt;1 March 2013,&lt;br/&gt;15:10
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Falcon_9_v1.0" title="Falcon 9 v1.0"&gt;F9 v1.0&lt;/a&gt;&lt;sup class="reference" id="cite_ref-MuskMay2012_13-4"&gt;&lt;a href="#cite_note-MuskMay2012-13"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;7&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;B0007.1&lt;sup class="reference" id="cite_ref-block_numbers_14-4"&gt;&lt;a href="#cite_note-block_numbers-14"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;8&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Force_Station" title="Cape Canaveral Space Force Station"&gt;CCAFS&lt;/a&gt;,&lt;br/&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Launch_Complex_40" title="Cape Canaveral Space Launch Complex 40"&gt;SLC-40&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/SpaceX_CRS-2" title="SpaceX CRS-2"&gt;SpaceX CRS-2&lt;/a&gt;&lt;sup class="reference" id="cite_ref-sxManifest20120925_28-1"&gt;&lt;a href="#cite_note-sxManifest20120925-28"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;22&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;(Dragon C104)
    &lt;/td&gt;
    &lt;td&gt;4,877 kg (10,752 lb)
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Low_Earth_orbit" title="Low Earth orbit"&gt;LEO&lt;/a&gt; (&lt;a class="mw-redirect" href="/wiki/ISS" title="ISS"&gt;ISS&lt;/a&gt;)
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/NASA" title="NASA"&gt;NASA&lt;/a&gt; (&lt;a href="/wiki/Commercial_Resupply_Services" title="Commercial Resupply Services"&gt;CRS&lt;/a&gt;)
    &lt;/td&gt;
    &lt;td class="table-success" style="background: #9EFF9E; color:black; vertical-align: middle; text-align: center;"&gt;Success
    &lt;/td&gt;
    &lt;td class="table-noAttempt" style="background: #EEE; color:black; vertical-align: middle; white-space: nowrap; text-align: center;"&gt;No attempt
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;td colspan="9"&gt;Last launch of the original Falcon 9 v1.0 &lt;a href="/wiki/Launch_vehicle" title="Launch vehicle"&gt;launch vehicle&lt;/a&gt;, first use of the unpressurized trunk section of Dragon.&lt;sup class="reference" id="cite_ref-sxf9_20110321_35-0"&gt;&lt;a href="#cite_note-sxf9_20110321-35"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;29&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;th rowspan="2" scope="row" style="text-align:center;"&gt;6
    &lt;/th&gt;
    &lt;td&gt;29 September 2013,&lt;br/&gt;16:00&lt;sup class="reference" id="cite_ref-pa20130930_36-0"&gt;&lt;a href="#cite_note-pa20130930-36"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;30&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Falcon_9_v1.1" title="Falcon 9 v1.1"&gt;F9 v1.1&lt;/a&gt;&lt;sup class="reference" id="cite_ref-MuskMay2012_13-5"&gt;&lt;a href="#cite_note-MuskMay2012-13"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;7&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;br/&gt;B1003&lt;sup class="reference" id="cite_ref-block_numbers_14-5"&gt;&lt;a href="#cite_note-block_numbers-14"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;8&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a class="mw-redirect" href="/wiki/Vandenberg_Air_Force_Base" title="Vandenberg Air Force Base"&gt;VAFB&lt;/a&gt;,&lt;br/&gt;&lt;a href="/wiki/Vandenberg_Space_Launch_Complex_4" title="Vandenberg Space Launch Complex 4"&gt;SLC-4E&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/CASSIOPE" title="CASSIOPE"&gt;CASSIOPE&lt;/a&gt;&lt;sup class="reference" id="cite_ref-sxManifest20120925_28-2"&gt;&lt;a href="#cite_note-sxManifest20120925-28"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;22&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;sup class="reference" id="cite_ref-CASSIOPE_MDA_37-0"&gt;&lt;a href="#cite_note-CASSIOPE_MDA-37"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;31&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;500 kg (1,100 lb)
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Polar_orbit" title="Polar orbit"&gt;Polar orbit&lt;/a&gt; &lt;a href="/wiki/Low_Earth_orbit" title="Low Earth orbit"&gt;LEO&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Maxar_Technologies" title="Maxar Technologies"&gt;MDA&lt;/a&gt;
    &lt;/td&gt;
    &lt;td class="table-success" style="background: #9EFF9E; color:black; vertical-align: middle; text-align: center;"&gt;Success&lt;sup class="reference" id="cite_ref-pa20130930_36-1"&gt;&lt;a href="#cite_note-pa20130930-36"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;30&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td class="table-no2" style="background: #FFE3E3; color: black; vertical-align: middle; text-align: center;"&gt;Uncontrolled&lt;br/&gt;&lt;small&gt;(ocean)&lt;/small&gt;&lt;sup class="reference" id="cite_ref-ocean_landing_38-0"&gt;&lt;a href="#cite_note-ocean_landing-38"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;d&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;td colspan="9"&gt;First commercial mission with a private customer, first launch from Vandenberg, and demonstration flight of Falcon 9 v1.1 with an improved 13-tonne to LEO capacity.&lt;sup class="reference" id="cite_ref-sxf9_20110321_35-1"&gt;&lt;a href="#cite_note-sxf9_20110321-35"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;29&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; After separation from the second stage carrying Canadian commercial and scientific satellites, the first stage booster performed a controlled reentry,&lt;sup class="reference" id="cite_ref-39"&gt;&lt;a href="#cite_note-39"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;32&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; and an &lt;a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests"&gt;ocean touchdown test&lt;/a&gt; for the first time. This provided good test data, even though the booster started rolling as it neared the ocean, leading to the shutdown of the central engine as the roll depleted it of fuel, resulting in a hard impact with the ocean.&lt;sup class="reference" id="cite_ref-pa20130930_36-2"&gt;&lt;a href="#cite_note-pa20130930-36"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;30&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; This was the first known attempt of a rocket engine being lit to perform a supersonic retro propulsion, and allowed SpaceX to enter a public-private partnership with &lt;a href="/wiki/NASA" title="NASA"&gt;NASA&lt;/a&gt; and its Mars entry, descent, and landing technologies research projects.&lt;sup class="reference" id="cite_ref-40"&gt;&lt;a href="#cite_note-40"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;33&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; &lt;small&gt;(&lt;a href="#Maiden_flight_of_v1.1"&gt;more details below&lt;/a&gt;)&lt;/small&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;th rowspan="2" scope="row" style="text-align:center;"&gt;7
    &lt;/th&gt;
    &lt;td&gt;3 December 2013,&lt;br/&gt;22:41&lt;sup class="reference" id="cite_ref-sfn_wwls20130624_41-0"&gt;&lt;a href="#cite_note-sfn_wwls20130624-41"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;34&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Falcon_9_v1.1" title="Falcon 9 v1.1"&gt;F9 v1.1&lt;/a&gt;&lt;br/&gt;B1004
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Force_Station" title="Cape Canaveral Space Force Station"&gt;CCAFS&lt;/a&gt;,&lt;br/&gt;&lt;a href="/wiki/Cape_Canaveral_Space_Launch_Complex_40" title="Cape Canaveral Space Launch Complex 40"&gt;SLC-40&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/SES-8" title="SES-8"&gt;SES-8&lt;/a&gt;&lt;sup class="reference" id="cite_ref-sxManifest20120925_28-3"&gt;&lt;a href="#cite_note-sxManifest20120925-28"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;22&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;sup class="reference" id="cite_ref-spx-pr_42-0"&gt;&lt;a href="#cite_note-spx-pr-42"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;35&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;&lt;sup class="reference" id="cite_ref-aw20110323_43-0"&gt;&lt;a href="#cite_note-aw20110323-43"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;36&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td&gt;3,170 kg (6,990 lb)
    &lt;/td&gt;
    &lt;td&gt;&lt;a href="/wiki/Geostationary_transfer_orbit" title="Geostationary transfer orbit"&gt;GTO&lt;/a&gt;
    &lt;/td&gt;
    &lt;td&gt;&lt;a class="mw-redirect" href="/wiki/SES_S.A." title="SES S.A."&gt;SES&lt;/a&gt;
    &lt;/td&gt;
    &lt;td class="table-success" style="background: #9EFF9E; color:black; vertical-align: middle; text-align: center;"&gt;Success&lt;sup class="reference" id="cite_ref-SNMissionStatus7_44-0"&gt;&lt;a href="#cite_note-SNMissionStatus7-44"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;37&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;
    &lt;td class="table-noAttempt" style="background: #EEE; color:black; vertical-align: middle; white-space: nowrap; text-align: center;"&gt;No attempt&lt;br/&gt;&lt;sup class="reference" id="cite_ref-sf10120131203_45-0"&gt;&lt;a href="#cite_note-sf10120131203-45"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;38&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt;
    &lt;/td&gt;&lt;/tr&gt;
    &lt;tr&gt;
    &lt;td colspan="9"&gt;First &lt;a href="/wiki/Geostationary_transfer_orbit" title="Geostationary transfer orbit"&gt;Geostationary transfer orbit&lt;/a&gt; (GTO) launch for Falcon 9,&lt;sup class="reference" id="cite_ref-spx-pr_42-1"&gt;&lt;a href="#cite_note-spx-pr-42"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;35&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; and first successful reignition of the second stage.&lt;sup class="reference" id="cite_ref-46"&gt;&lt;a href="#cite_note-46"&gt;&lt;span class="cite-bracket"&gt;[&lt;/span&gt;39&lt;span class="cite-bracket"&gt;]&lt;/span&gt;&lt;/a&gt;&lt;/sup&gt; SES-8 was inserted into a &lt;a href="/wiki/Geostationary_transfer_orbit" title="Geostationary transfer orbit"&gt;Super-Synchronous Transfer Orbit&lt;/a&gt; of 79,341 km (49,300 mi) in apogee with an &lt;a href="/wiki/Orbital_inclination" title="Orbital inclination"&gt;inclination&lt;/a&gt; of 20.55° to the &lt;a href="/wiki/Equator" title="Equator"&gt;equator&lt;/a&gt;.
    &lt;/td&gt;&lt;/tr&gt;&lt;/tbody&gt;&lt;/table&gt;


You should able to see the columns names embedded in the table header elements `&lt;th&gt;` as follows:


```
&lt;tr&gt;
&lt;th scope="col"&gt;Flight No.
&lt;/th&gt;
&lt;th scope="col"&gt;Date and&lt;br/&gt;time (&lt;a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time"&gt;UTC&lt;/a&gt;)
&lt;/th&gt;
&lt;th scope="col"&gt;&lt;a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters"&gt;Version,&lt;br/&gt;Booster&lt;/a&gt; &lt;sup class="reference" id="cite_ref-booster_11-0"&gt;&lt;a href="#cite_note-booster-11"&gt;[b]&lt;/a&gt;&lt;/sup&gt;
&lt;/th&gt;
&lt;th scope="col"&gt;Launch site
&lt;/th&gt;
&lt;th scope="col"&gt;Payload&lt;sup class="reference" id="cite_ref-Dragon_12-0"&gt;&lt;a href="#cite_note-Dragon-12"&gt;[c]&lt;/a&gt;&lt;/sup&gt;
&lt;/th&gt;
&lt;th scope="col"&gt;Payload mass
&lt;/th&gt;
&lt;th scope="col"&gt;Orbit
&lt;/th&gt;
&lt;th scope="col"&gt;Customer
&lt;/th&gt;
&lt;th scope="col"&gt;Launch&lt;br/&gt;outcome
&lt;/th&gt;
&lt;th scope="col"&gt;&lt;a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests"&gt;Booster&lt;br/&gt;landing&lt;/a&gt;
&lt;/th&gt;&lt;/tr&gt;
```



```python
column_names = []

# 找到第一个表格
first_launch_table = tables[2] # 假设你想要第一个表格

# 使用 find_all() 函数找到所有 th 元素
headers = first_launch_table.find_all('th')

# 迭代每个 th 元素并提取列名
for header in headers:
    column_name = extract_column_from_header(header)
    if column_name:  # 仅添加非空列名
        column_names.append(column_name)

print(column_names)
```

    ['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


Next, we just need to iterate through the `&lt;th&gt;` elements and apply the provided `extract_column_from_header()` to extract column name one by one



```python
print(column_names)
```

    ['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe



```python
launch_dict= dict.fromkeys(column_names)

# Remove an irrelvant column
del launch_dict['Date and time ( )']

# Let's initial the launch_dict with each value to be an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]
```

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.



```python
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 辅助函数定义（示例）
def date_time(date_time_str):
    # 假设这个函数将日期和时间字符串分开
    return date_time_str.split()  # 示例：按空格分割

def booster_version(version_str):
    # 假设这个函数处理火箭版本
    return version_str.strip()  # 示例：去除空格

def get_mass(mass_str):
    # 假设这个函数从字符串中提取有效的质量
    return mass_str.strip()  # 示例：去除空格

def landing_status(status_str):
    # 假设这个函数处理着陆状态
    return status_str.strip()  # 示例：去除空格

# 指定维基百科页面的 URL
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&amp;oldid=1027686922"

# 发送 GET 请求
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 创建初始字典
    launch_dict = {
        'Flight No.': [],
        'Date': [],
        'Time': [],
        'Version, Booster': [],
        'Launch site': [],
        'Payload': [],
        'Payload mass': [],
        'Orbit': [],
        'Customer': [],
        'Launch outcome': [],
        'Booster landing': []
    }
    
    extracted_row = 0
    
    # 提取每个表格
    for table in soup.find_all('table', "wikitable plainrowheaders collapsible"):
        # 获取表格行
        for rows in table.find_all("tr"):
            # 检查第一列是否为数字
            if rows.th:
                if rows.th.string:
                    flight_number = rows.th.string.strip()
                    flag = flight_number.isdigit()
                else:
                    flag = False
            else:
                flag = False
            
            # 获取表格元素
            row = rows.find_all('td')
            # 如果是数字，保存单元格到字典中
            if flag:
                extracted_row += 1
                
                # Flight Number value
                launch_dict['Flight No.'].append(flight_number)
                
                # Date and Time values
                datatimelist = date_time(row[0].text)
                
                # Date value
                date = datatimelist[0].strip(',')
                launch_dict['Date'].append(date)
                
                # Time value
                time = datatimelist[1]
                launch_dict['Time'].append(time)
                
                # Booster version
                bv = booster_version(row[1].text)
                if not bv:
                    bv = row[1].a.string if row[1].a else ''
                launch_dict['Version, Booster'].append(bv)
                
                # Launch Site
                launch_site = row[2].a.string if row[2].a else ''
                launch_dict['Launch site'].append(launch_site)
                
                # Payload
                payload = row[3].a.string if row[3].a else ''
                launch_dict['Payload'].append(payload)
                
                # Payload Mass
                payload_mass = get_mass(row[4].text)
                launch_dict['Payload mass'].append(payload_mass)
                
                # Orbit
                orbit = row[5].a.string if row[5].a else ''
                launch_dict['Orbit'].append(orbit)
                
                # Customer
                customer = row[6].a.string if row[6].a else ''
                launch_dict['Customer'].append(customer)
                
                # Launch outcome
                launch_outcome = list(row[7].strings)[0] if row[7].strings else ''
                launch_dict['Launch outcome'].append(launch_outcome)
                
                # Booster landing
                booster_landing = landing_status(row[8].text)
                launch_dict['Booster landing'].append(booster_landing)

    # 打印结果
    print("填充后的字典:")
    print(launch_dict)

else:
    print(f"请求失败，状态码: {response.status_code}")
```

    填充后的字典:
    {'Flight No.': ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121'], 'Date': ['4', '8', '22', '8', '1', '29', '3', '6', '18', '14', '5', '7', '21', '10', '11', '2', '14', '27', '28', '22', '17', '4', '8', '6', '27', '15', '18', '14', '14', '19', '16', '30', '1', '15', '3', '23', '25', '5', '14', '24', '7', '9', '11', '30', '15', '23', '8', '31', '22', '6', '30', '2', '18', '11', '22', '4', '29', '22', '25', '7', '10', '8', '15', '3', '5', '23', '11', '22', '2', '4', '24', '12', '25', '6', '11', '5', '17', '7', '19', '29', '17', '7', '18', '22', '30', '4', '13', '30', '20', '7', '18', '30', '3', '6', '18', '24', '5', '16', '21', '25', '6', '13', '19', '8', '20', '24', '4', '16', '4', '11', '14', '24', '7', '23', '29', '4', '9', '15', '26', '3', '6'], 'Time': ['June', 'December', 'May', 'October', 'March', 'September', 'December', 'January', 'April', 'July', 'August', 'September', 'September', 'January', 'February', 'March', 'April', 'April', 'June', 'December', 'January', 'March', 'April', 'May', 'May', 'June', 'July', 'August', 'January', 'February', 'March', 'March', 'May', 'May', 'June', 'June', 'June', 'July', 'August', 'August', 'September', 'October', 'October', 'October', 'December', 'December', 'January', 'January', 'February', 'March', 'March', 'April', 'April', 'May', 'May', 'June', 'June', 'July', 'July', 'August', 'September', 'October', 'November', 'December', 'December', 'December', 'January', 'February', 'March', 'May', 'May', 'June', 'July', 'August', 'November', 'December', 'December', 'January', 'January', 'January', 'February', 'March', 'March', 'April', 'May', 'June', 'June', 'June', 'July', 'August', 'August', 'August', 'September', 'October', 'October', 'October', 'November', 'November', 'November', 'November', 'December', 'December', 'December', 'January', 'January', 'January', 'February', 'February', 'March', 'March', 'March', 'March', 'April', 'April', 'April', 'May', 'May', 'May', 'May', 'June', 'June'], 'Version, Booster': ['F9 v1.0[7]B0003.1[8]', 'F9 v1.0[7]B0004.1[8]', 'F9 v1.0[7]B0005.1[8]', 'F9 v1.0[7]B0006.1[8]', 'F9 v1.0[7]B0007.1[8]', 'F9 v1.1[7]B1003[8]', 'F9 v1.1B1004', 'F9 v1.1', 'F9 v1.1', 'F9 v1.1', 'F9 v1.1', 'F9 v1.1B1011[8]', 'F9 v1.1B1010[8]', 'F9 v1.1B1012[8]', 'F9 v1.1B1013[8]', 'F9 v1.1B1014[8]', 'F9 v1.1B1015[8]', 'F9 v1.1B1016[8]', 'F9 v1.1B1018[8]', 'F9 FTB1019.1[98]', 'F9 v1.1B1017[8]', 'F9 FTB1020.1[108]', 'F9 FTB1021.1[115]', 'F9 FTB1022.1[122]', 'F9 FTB1023.1[129]', 'F9 FTB1024.1[108]', 'F9 FTB1025.1[129]', 'F9 FTB1026.1[108]', 'F9 FTB1029.1[151]', 'F9 FTB1031.1[8]', 'F9 FTB1030.1[163]', 'F9 FT ♺B1021.2[115]', 'F9 FTB1032.1[129]', 'F9 FTB1034.1[179]', 'F9 FTB1035.1[185]', 'F9 FT ♺ B1029.2[195]', 'F9 FTB1036.1[198]', 'F9 FTB1037.1[200]', 'F9 B4B1039.1[207]', 'F9 FTB1038.1[209]', 'F9 B4B1040.1[108]', 'F9 B4B1041.1[219]', 'F9 FT ♺ B1031.2[220]', 'F9 B4B1042.1[219]', 'F9 FT ♺ B1035.2[227]', 'F9 FT ♺ B1036.2[227]', 'F9 B4B1043.1[238]', 'F9 FT ♺ B1032.2[245]', 'F9 FT ♺B1038.2[268]', 'F9 B4B1044.1[108]', 'F9 B4 ♺B1041.2[268]', 'F9 B4 ♺B1039.2[292]', 'F9 B4B1045.1[268]', 'F9 B5[311]B1046.1[268]', 'F9 B4 ♺B1043.2[322]', 'F9 B4 ♺B1040.2[268]', 'F9 B4 ♺B1045.2[336]', 'F9 B5B1047.1', 'F9 B5[349]B1048.1[350]', 'F9 B5 ♺ B1046.2[354]', 'F9 B5B1049.1[268]', 'F9 B5 ♺B1048.2[364]', 'F9 B5 ♺B1047.2[268]', 'F9 B5 ♺ B1046.3[268]SHERPA', 'F9 B5B1050[268]', 'F9 B5B1054[387]', 'F9 B5 ♺B1049.2[397]', 'F9 B5 ♺B1048.3[399]', 'F9 B5B1051.1[268][413]', 'F9 B5B1056.1[420]', 'F9 B5 ♺B1049.3[434]', 'F9 B5 ♺B1051.2[420]', 'F9 B5 ♺B1056.2[465]', 'F9 B5 ♺B1047.3[472]', 'F9 B5 ♺B1048.4', 'F9 B5B1059.1[482]', 'F9 B5 ♺B1056.3[482]', 'F9 B5 ♺ B1049.4', 'F9 B5 ♺ B1046.4', 'F9 B5 ♺B1051.3', 'F9 B5 ♺B1056.4', 'F9 B5 ♺B1059.2', 'F9 B5 ♺ B1048.5', 'F9 B5 ♺B1051.4', 'F9 B5B1058.1[518]', 'F9 B5 ♺B1049.5', 'F9 B5 ♺B1059.3', 'F9 B5B1060.1', 'F9 B5 ♺ B1058.2[544]', 'F9 B5 ♺B1051.5', 'F9 B5 ♺B1049.6[544]', 'F9 B5 ♺B1059.4', 'F9 B5 ♺B1060.2[563]', 'F9 B5 ♺B1058.3[565]', 'F9 B5 ♺B1051.6[568]', 'F9 B5 ♺B1060.3', 'F9 B5B1062.1', 'F9 B5B1061.1[582]', 'F9 B5B1063.1', 'F9 B5 ♺B1049.7[590]', 'F9 B5 ♺B1058.4[592]', 'F9 B5 ♺B1051.7', 'F9 B5 ♺B1059.5', 'F9 B5 ♺B1060.4', 'F9 B5 ♺B1051.8[609]', 'F9 B5 ♺B1058.5[613]', 'F9 B5 ♺B1060.5[624]', 'F9 B5 ♺B1059.6', 'F9 B5 ♺B1049.8[634]', 'F9 B5 ♺B1058.6[638]', 'F9 B5 ♺B1051.9', 'F9 B5 ♺B1060.6[643]', 'F9 B5 ♺B1058.7', 'F9 B5 ♺B1061.2[647]', 'F9 B5 ♺B1060.7[652]', 'F9 B5 ♺B1049.9[655]', 'F9 B5 ♺B1051.10[657]', 'F9 B5 ♺B1058.8[660]', 'F9 B5 ♺B1063.2[665]', 'F9 B5 B1067.1[668]', 'F9 B5 ♺ B1061.3'], 'Launch site': ['CCAFS', 'CCAFS', 'CCAFS', 'CCAFS', 'CCAFS', 'VAFB', 'CCAFS', 'CCAFS', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'VAFB', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'Cape Canaveral', 'VAFB', 'KSC', 'KSC', 'KSC', 'KSC', 'KSC', 'KSC', 'KSC', 'VAFB', 'KSC', 'KSC', 'VAFB', 'KSC', 'VAFB', 'KSC', 'KSC', 'Cape Canaveral', 'VAFB', 'CCAFS', 'CCAFS', 'VAFB', 'CCAFS', 'VAFB', 'CCAFS', 'CCAFS', 'KSC', 'VAFB', 'CCAFS', 'CCAFS', 'CCAFS', 'VAFB', 'CCAFS', 'CCAFS', 'VAFB', 'KSC', 'VAFB', 'CCAFS', 'CCAFS', 'VAFB', 'CCAFS', 'KSC', 'CCAFS', 'CCAFS', 'VAFB', 'CCAFS', 'CCAFS', 'CCAFS', 'CCAFS', 'CCAFS', 'CCAFS', 'KSC', 'CCAFS', 'CCAFS', 'CCAFS', 'KSC', 'KSC', 'KSC', 'CCAFS', 'CCAFS', 'CCAFS', 'CCAFS', 'KSC', 'CCAFS', 'CCAFS', 'KSC', 'KSC', 'KSC', 'CCAFS', 'CCAFS', 'KSC', 'VAFB', 'CCAFS', 'KSC', 'CCSFS', 'KSC', 'CCSFS', 'KSC', 'CCSFS', 'CCSFS', 'CCSFS', 'KSC', 'CCSFS', 'KSC', 'CCSFS', 'CCSFS', 'KSC', 'CCSFS', 'KSC', 'CCSFS', 'KSC', 'CCSFS', 'KSC', 'CCSFS'], 'Payload': ['Dragon Spacecraft Qualification Unit', 'Dragon', 'Dragon', 'SpaceX CRS-1', 'SpaceX CRS-2', 'CASSIOPE', 'SES-8', 'Thaicom 6', 'SpaceX CRS-3', 'Orbcomm-OG2', 'AsiaSat 8', 'AsiaSat 6', 'SpaceX CRS-4', 'SpaceX CRS-5', 'DSCOVR', 'ABS-3A', 'SpaceX CRS-6', 'TürkmenÄlem 52°E / MonacoSAT', 'SpaceX CRS-7', 'Orbcomm-OG2', 'Jason-3', 'SES-9', 'SpaceX CRS-8', 'JCSAT-14', 'Thaicom 8', 'ABS-2A', 'SpaceX CRS-9', 'JCSAT-16', 'Iridium NEXT', 'SpaceX CRS-10', 'EchoStar 23', 'SES-10', 'NROL-76', 'Inmarsat-5 F4', 'SpaceX CRS-11', 'BulgariaSat-1', 'Iridium NEXT', 'Intelsat 35e', 'SpaceX CRS-12', 'Formosat-5', 'Boeing X-37B', 'Iridium NEXT', 'SES-11', 'Koreasat 5A', 'SpaceX CRS-13', 'Iridium NEXT', 'Zuma', 'GovSat-1', 'Paz', 'Hispasat 30W-6', 'Iridium NEXT', 'SpaceX CRS-14', 'Transiting Exoplanet Survey Satellite', 'Bangabandhu-1', 'Iridium NEXT', 'SES-12', 'SpaceX CRS-15', 'Telstar 19V', 'Iridium NEXT', 'Merah Putih', 'Telstar 18V', 'SAOCOM 1A', "Es'hail 2", 'SSO-A', 'SpaceX CRS-16', 'GPS III', 'Iridium NEXT', 'Nusantara Satu', 'Crew Dragon Demo-1', 'SpaceX CRS-17', 'Starlink', 'RADARSAT Constellation', 'SpaceX CRS-18', 'AMOS-17', 'Starlink', 'SpaceX CRS-19', 'JCSat-18', 'Starlink', 'Crew Dragon in-flight abort test', 'Starlink', 'Starlink', 'SpaceX CRS-20', 'Starlink', 'Starlink', 'Crew Dragon Demo-2', 'Starlink', 'Starlink', 'GPS III', 'ANASIS-II', 'Starlink', 'Starlink', 'SAOCOM 1B', 'Starlink', 'Starlink', 'Starlink', 'Starlink', 'GPS III', 'Crew-1', 'Sentinel-6 Michael Freilich (Jason-CS A)', 'Starlink', 'SpaceX CRS-21', 'SXM-7', 'NROL-108', 'Türksat 5A', 'Starlink', 'Transporter-1', 'Starlink', 'Starlink', 'Starlink', 'Starlink', 'Starlink', 'Starlink', 'Starlink', 'Crew-2', 'Starlink', 'Starlink', 'Starlink', 'Starlink', 'Starlink', 'SpaceX CRS-22', 'SXM-8'], 'Payload mass': ['', '', '525\xa0kg (1,157\xa0lb)[19]', '4,700\xa0kg (10,400\xa0lb)', '4,877\xa0kg (10,752\xa0lb)', '500\xa0kg (1,100\xa0lb)', '3,170\xa0kg (6,990\xa0lb)', '3,325\xa0kg (7,330\xa0lb)', '2,296\xa0kg (5,062\xa0lb)[45]', '1,316\xa0kg (2,901\xa0lb)', '4,535\xa0kg (9,998\xa0lb)', '4,428\xa0kg (9,762\xa0lb)', '2,216\xa0kg (4,885\xa0lb)[62]', '2,395\xa0kg (5,280\xa0lb)[69]', '570\xa0kg (1,260\xa0lb)', '4,159\xa0kg (9,169\xa0lb)', '1,898\xa0kg (4,184\xa0lb)[82]', '4,707\xa0kg (10,377\xa0lb)', '1,952\xa0kg (4,303\xa0lb)[92]', '2,034\xa0kg (4,484\xa0lb)', '553\xa0kg (1,219\xa0lb)', '5,271\xa0kg (11,621\xa0lb)', '3,136\xa0kg (6,914\xa0lb)[116]', '4,696\xa0kg (10,353\xa0lb)[124]', '3,100\xa0kg (6,800\xa0lb)[132]', '3,600\xa0kg (7,900\xa0lb)', '2,257\xa0kg (4,976\xa0lb)[142]', '4,600\xa0kg (10,100\xa0lb)', '9,600\xa0kg (21,200\xa0lb)', '2,490\xa0kg (5,490\xa0lb)[160]', '5,600\xa0kg (12,300\xa0lb)[164]', '5,300\xa0kg (11,700\xa0lb)[169]', 'Classified', '6,070\xa0kg (13,380\xa0lb)[181]', '2,708\xa0kg (5,970\xa0lb)[186]', '3,669\xa0kg (8,089\xa0lb)[197]', '9,600\xa0kg (21,200\xa0lb)', '6,761\xa0kg (14,905\xa0lb)[202]', '3,310\xa0kg (7,300\xa0lb)', '475\xa0kg (1,047\xa0lb)[212]', '4,990\xa0kg (11,000\xa0lb)[216]+ OTV payload', '9,600\xa0kg (21,200\xa0lb)', '5,200\xa0kg (11,500\xa0lb)', '3,500\xa0kg (7,700\xa0lb)', '2,205\xa0kg (4,861\xa0lb)', '9,600\xa0kg (21,200\xa0lb)', 'Classified', '4,230\xa0kg (9,330\xa0lb)[247]', '2,150\xa0kg (4,740\xa0lb)', '6,092\xa0kg (13,431\xa0lb)[279]', '9,600\xa0kg (21,200\xa0lb)', '2,647\xa0kg (5,836\xa0lb)[292]', '362\xa0kg (798\xa0lb)[305]', '3,600\xa0kg (7,900\xa0lb)[314]', '6,460\xa0kg (14,240\xa0lb)[g]', '5,384\xa0kg (11,870\xa0lb)[332]', '2,697\xa0kg (5,946\xa0lb)[337]', '7,075\xa0kg (15,598\xa0lb)[342]', '9,600\xa0kg (21,200\xa0lb)', '5,800\xa0kg (12,800\xa0lb)[357]', '7,060\xa0kg (15,560\xa0lb)[361]', '3,000\xa0kg (6,600\xa0lb)[363]', '5,300\xa0kg (11,700\xa0lb)[369]', '~4,000\xa0kg (8,800\xa0lb)[372]', '2,500\xa0kg (5,500\xa0lb)[382]', '4,400\xa0kg (9,700\xa0lb)[388]', '9,600\xa0kg (21,200\xa0lb)', '4,850\xa0kg (10,690\xa0lb)[403]', '12,055\xa0kg (26,577\xa0lb)[415][h]', '2,495\xa0kg (5,501\xa0lb)[432]', '13,620\xa0kg (30,030\xa0lb)[5]', '4,200\xa0kg (9,300\xa0lb)[443]', '2,268\xa0kg (5,000\xa0lb)[464]', '6,500\xa0kg (14,300\xa0lb)[474]', '15,600\xa0kg (34,400\xa0lb)[5]', '2,617\xa0kg (5,769\xa0lb)', '6,956\xa0kg (15,335\xa0lb)[486]', '15,600\xa0kg (34,400\xa0lb)[5]', '12,050\xa0kg (26,570\xa0lb)', '15,600\xa0kg (34,400\xa0lb)[5]', '15,600\xa0kg (34,400\xa0lb)[5]', '1,977\xa0kg (4,359\xa0lb)[507]', '15,600\xa0kg (34,400\xa0lb)[5]', '15,600\xa0kg (34,400\xa0lb)[5]', '12,530\xa0kg (27,620\xa0lb)[519]', '15,600\xa0kg (34,400\xa0lb)[5]', '15,410\xa0kg (33,970\xa0lb)[523]', '4,311\xa0kg (9,504\xa0lb)[530]', '5,000–6,000\xa0kg (11,000–13,000\xa0lb)', '14,932\xa0kg (32,919\xa0lb)', '~15,440\xa0kg (34,040\xa0lb)', '3,130\xa0kg (6,900\xa0lb)[558]', '15,600\xa0kg (34,400\xa0lb)[5]', '15,600\xa0kg (34,400\xa0lb)[5]', '15,600\xa0kg (34,400\xa0lb)[5]', '15,600\xa0kg (34,400\xa0lb)', '4,311\xa0kg (9,504\xa0lb)', '~12,500\xa0kg (27,600\xa0lb)', '1,192\xa0kg (2,628\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '2,972\xa0kg (6,552\xa0lb)', '7,000\xa0kg (15,000\xa0lb)', 'Classified', '3,500\xa0kg (7,700\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '~5,000\xa0kg (11,000\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '~13,000\xa0kg (29,000\xa0lb)[648]', '15,600\xa0kg (34,400\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '~14,000\xa0kg (31,000\xa0lb)', '15,600\xa0kg (34,400\xa0lb)', '3,328\xa0kg (7,337\xa0lb)', '7,000\xa0kg (15,000\xa0lb)'], 'Orbit': ['LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'Polar orbit', 'GTO', 'GTO', 'LEO', 'LEO', 'GTO', 'GTO', 'LEO', 'LEO', 'HEO', 'GTO', 'LEO', 'GTO', 'LEO', 'LEO', 'LEO', 'GTO', 'LEO', 'GTO', 'GTO', 'GTO', 'LEO', 'GTO', 'Polar', 'LEO', 'GTO', 'GTO', 'LEO', 'GTO', 'LEO', 'GTO', 'LEO', 'GTO', 'LEO', 'SSO', 'LEO', 'Polar', 'GTO', 'GTO', 'LEO', 'Polar', 'LEO', 'GTO', 'SSO', 'GTO', 'Polar', 'LEO', 'HEO', 'GTO', 'Polar', 'GTO', 'LEO', 'GTO', 'Polar', 'GTO', 'GTO', 'SSO', 'GTO', 'SSO', 'LEO', 'MEO', 'Polar', 'GTO', 'LEO', 'LEO', 'LEO', 'SSO', 'LEO', 'GTO', 'LEO', 'LEO', 'GTO', 'LEO', 'Sub-orbital', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'MEO', 'GTO', 'LEO', 'LEO', 'SSO', 'LEO', 'LEO', 'LEO', 'LEO', 'MEO', 'LEO', 'LEO', 'LEO', 'LEO', 'GTO', 'LEO', 'GTO', 'LEO', 'SSO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'LEO', 'GTO'], 'Customer': ['SpaceX', 'NASA', 'NASA', 'NASA', 'NASA', 'MDA', 'SES', 'Thaicom', 'NASA', 'Orbcomm', 'AsiaSat', 'AsiaSat', 'NASA', 'NASA', 'USAF', 'ABS', 'NASA', None, 'NASA', 'Orbcomm', 'NASA', 'SES', 'NASA', 'SKY Perfect JSAT Group', 'Thaicom', 'ABS', 'NASA', 'SKY Perfect JSAT Group', 'Iridium Communications', 'NASA', 'EchoStar', 'SES', 'NRO', 'Inmarsat', 'NASA', 'Bulsatcom', 'Iridium Communications', 'Intelsat', 'NASA', 'NSPO', 'USAF', 'Iridium Communications', 'SES S.A.', 'KT Corporation', 'NASA', 'Iridium Communications', 'Northrop Grumman', 'SES', 'Hisdesat', 'Hispasat', 'Iridium Communications', 'NASA', 'NASA', 'Thales-Alenia', 'Iridium Communications', 'SES', 'NASA', 'Telesat', 'Iridium Communications', 'Telkom Indonesia', 'Telesat', 'CONAE', "Es'hailSat", 'Spaceflight Industries', 'NASA', 'USAF', 'Iridium Communications', 'PSN', 'NASA', 'NASA', 'SpaceX', 'Canadian Space Agency', 'NASA', 'Spacecom', 'SpaceX', 'NASA', 'Sky Perfect JSAT', 'SpaceX', 'NASA', 'SpaceX', 'SpaceX', 'NASA', 'SpaceX', 'SpaceX', 'NASA', 'SpaceX', 'SpaceX', 'U.S. Space Force', 'Republic of Korea Army', 'SpaceX', 'SpaceX', 'CONAE', 'SpaceX', 'SpaceX', 'SpaceX', 'SpaceX', 'USSF', 'NASA', 'NASA', 'SpaceX', 'NASA', 'Sirius XM', 'NRO', 'Türksat', 'SpaceX', '', 'SpaceX', 'SpaceX', 'SpaceX', 'SpaceX', 'SpaceX', 'SpaceX', 'SpaceX', 'NASA', 'SpaceX', 'SpaceX', 'SpaceX', 'SpaceX', 'SpaceX', 'NASA', 'Sirius XM'], 'Launch outcome': ['Success\n', 'Success', 'Success', 'Success\n', 'Success\n', 'Success', 'Success', 'Success', 'Success\n', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Failure', 'Success\n', 'Success\n', 'Success\n', 'Success', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success\n', 'Success', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n', 'Success\n'], 'Booster landing': ['Failure[9][10](parachute)', 'Failure[9][14](parachute)', 'No attempt', 'No attempt', 'No attempt', 'Uncontrolled(ocean)[d]', 'No attempt[38]', 'No attempt[43]', 'Controlled(ocean) [d][46]', 'Controlled(ocean)[d][46]', 'No attempt[57]', 'No attempt', 'Uncontrolled(ocean)[d][64]', 'Failure (drone ship)', 'Controlled(ocean)[d]', 'No attempt[77]', 'Failure[83](drone ship)', 'No attempt[89]', 'Precluded[94](drone ship)', 'Success[99](ground pad)', 'Failure(drone ship)', 'Failure(drone ship)', 'Success[118](drone ship)', 'Success(drone ship)', 'Success[133](drone ship)', 'Failure[64](drone ship)', 'Success(ground pad)', 'Success(drone ship)', 'Success[154](drone ship)', 'Success(ground pad)', 'No attempt[165]', 'Success(drone ship)', 'Success(ground pad)', 'No attempt[165]', 'Success(ground pad)', 'Success(drone ship)', 'Success(drone ship)', 'No attempt[165]', 'Success(ground pad)', 'Success(drone ship)', 'Success(ground pad)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(ground pad)', 'Controlled(ocean)[d][230]', 'Success(ground pad)', 'Controlled(ocean)[d][248]', 'No attempt[271]', 'No attempt[281]', 'No attempt[287]', 'No attempt[294]', 'Success[306](drone ship)', 'Success[315](drone ship)', 'No attempt[165]', 'No attempt[165]', 'No attempt[165]', 'Success[344](drone ship)', 'Success[352](drone ship)', 'Success[358](drone ship)', 'Success[361](drone ship)', 'Success[363](ground pad)', 'Success[370](drone ship)', 'Success[373](drone ship)', 'Failure[383](ground pad)', 'No attempt[386]', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(ground pad)', 'Success(ground pad)', 'No attempt[474]', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'No attempt', 'Success(drone ship)', 'Failure(drone ship)', 'Success(ground pad)', 'Failure(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(ground pad)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(ground pad)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(ground pad)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Failure (drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)', 'Success(drone ship)']}


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


## Authors


We can now export it to a &lt;b&gt;CSV&lt;/b&gt; for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 



```python
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 辅助函数定义（示例）
def date_time(date_time_str):
    return date_time_str.split()  # 示例：按空格分割

def booster_version(version_str):
    return version_str.strip()  # 示例：去除空格

def get_mass(mass_str):
    return mass_str.strip()  # 示例：去除空格

def landing_status(status_str):
    return status_str.strip()  # 示例：去除空格

# 指定维基百科页面的 URL
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&amp;oldid=1027686922"

# 发送 GET 请求
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 创建初始字典
    launch_dict = {
        'Flight No.': [],
        'Date': [],
        'Time': [],
        'Version, Booster': [],
        'Launch site': [],
        'Payload': [],
        'Payload mass': [],
        'Orbit': [],
        'Customer': [],
        'Launch outcome': [],
        'Booster landing': []
    }
    
    extracted_row = 0
    
    # 提取每个表格
    for table in soup.find_all('table', "wikitable plainrowheaders collapsible"):
        for rows in table.find_all("tr"):
            if rows.th:
                flight_number = rows.th.string.strip() if rows.th.string else ''
                flag = flight_number.isdigit()
            else:
                flag = False
            
            row = rows.find_all('td')
            if flag:
                extracted_row += 1
                
                # 填充 launch_dict
                launch_dict['Flight No.'].append(flight_number)
                
                datatimelist = date_time(row[0].text)
                date = datatimelist[0].strip(',')
                launch_dict['Date'].append(date)
                
                time = datatimelist[1]
                launch_dict['Time'].append(time)
                
                bv = booster_version(row[1].text)
                launch_dict['Version, Booster'].append(bv)
                
                launch_site = row[2].a.string if row[2].a else ''
                launch_dict['Launch site'].append(launch_site)
                
                payload = row[3].a.string if row[3].a else ''
                launch_dict['Payload'].append(payload)
                
                payload_mass = get_mass(row[4].text)
                launch_dict['Payload mass'].append(payload_mass)
                
                orbit = row[5].a.string if row[5].a else ''
                launch_dict['Orbit'].append(orbit)
                
                customer = row[6].a.string if row[6].a else ''
                launch_dict['Customer'].append(customer)
                
                launch_outcome = list(row[7].strings)[0] if row[7].strings else ''
                launch_dict['Launch outcome'].append(launch_outcome)
                
                booster_landing = landing_status(row[8].text)
                launch_dict['Booster landing'].append(booster_landing)

    # 创建 Pandas DataFrame
    launch_df = pd.DataFrame(launch_dict)

    # 打印数据框架的前几行
    print("\n创建的数据框架:")
    print(launch_df.head())  # 打印前5行

else:
    print(f"请求失败，状态码: {response.status_code}")
```

    
    创建的数据框架:
      Flight No. Date      Time      Version, Booster Launch site  \
    0          1    4      June  F9 v1.0[7]B0003.1[8]       CCAFS   
    1          2    8  December  F9 v1.0[7]B0004.1[8]       CCAFS   
    2          3   22       May  F9 v1.0[7]B0005.1[8]       CCAFS   
    3          4    8   October  F9 v1.0[7]B0006.1[8]       CCAFS   
    4          5    1     March  F9 v1.0[7]B0007.1[8]       CCAFS   
    
                                    Payload           Payload mass Orbit Customer  \
    0  Dragon Spacecraft Qualification Unit                          LEO   SpaceX   
    1                                Dragon                          LEO     NASA   
    2                                Dragon  525 kg (1,157 lb)[19]   LEO     NASA   
    3                          SpaceX CRS-1   4,700 kg (10,400 lb)   LEO     NASA   
    4                          SpaceX CRS-2   4,877 kg (10,752 lb)   LEO     NASA   
    
      Launch outcome            Booster landing  
    0      Success\n  Failure[9][10](parachute)  
    1        Success  Failure[9][14](parachute)  
    2        Success                 No attempt  
    3      Success\n                 No attempt  
    4      Success\n                 No attempt  



```python

```

&lt;a href="https://www.linkedin.com/in/nayefaboutayoun/"&gt;Nayef Abou Tayoun&lt;/a&gt;


&lt;code&gt;df.to_csv('spacex_web_scraped.csv', index=False)&lt;/code&gt;



```python
# 过滤掉猎鹰 1 号的发射记录
launch_df_filtered = launch_df[launch_df['Flight No.'] != 'Falcon 1']

# 计算猎鹰 9 号的发射次数
falcon_9_launches_count = launch_df_filtered[launch_df_filtered['Version, Booster'].str.contains('Falcon 9')].shape[0]

print(f"除去猎鹰 1 号的发射后，猎鹰 9 号的发射次数为: {falcon_9_launches_count}")
```

    除去猎鹰 1 号的发射后，猎鹰 9 号的发射次数为: 0


&lt;a href="https://www.linkedin.com/in/yan-luo-96288783/"&gt;Yan Luo&lt;/a&gt;



```python
# 过滤掉猎鹰 1 号的发射记录
launch_df_filtered = launch_df[launch_df['Flight No.'] != 'Falcon 1']

# 计算不包含猎鹰 9 号的发射次数
not_falcon_9_launches_count = launch_df_filtered[~launch_df_filtered['Version, Booster'].str.contains('Falcon 9')].shape[0]

print(f"除去猎鹰 1 号的发射后，不包含猎鹰 9 号的发射次数为: {not_falcon_9_launches_count}")
```

    除去猎鹰 1 号的发射后，不包含猎鹰 9 号的发射次数为: 121


&lt;!--
## Change Log
--&gt;


&lt;!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
--&gt;


Copyright © 2021 IBM Corporation. All rights reserved.
</pre><br class="Apple-interchange-newline">
